In [1]:
library(data.table)
library(dplyr)
library(stringr)
library(readr)
library(tibble)
library(lubridate)

# ============================================================
# CENTER OF GROWING SEASON FROM HALF-HOURLY GPP
# Growing season: daily GPP > GPPmin + 0.3 * (GPPmax - GPPmin)
# ============================================================

# ------------------------------------------------------------
# 0) Paths
# ------------------------------------------------------------

base_dir <- "fluxnet_2017_2025_V02"

out_dir <- "derived_tables/outputs_afterEGU_results/center_growing_season"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

main_file <- "derived_tables/outputs_afterEGU_results/EFP_mortality_trait_hydro_combined.csv"

#---------------------------------------------------------
# 1) Load your modelling dataset
# ------------------------------------------------------------

model_dt <- fread(main_file)

# adapt if needed
if ("site_id" %in% names(model_dt) && !("SITE_ID" %in% names(model_dt))) {
  setnames(model_dt, "site_id", "SITE_ID")
}
needed_site_years <- unique(model_dt[, .(SITE_ID, YEAR)])
cat("Sites in modelling dataset:", uniqueN(needed_site_years$SITE_ID), "\n")
cat("Site-years in modelling dataset:", nrow(needed_site_years), "\n")



Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, isoyear, mday, minute, month, quarter, second, wday,
    week, yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




Sites in modelling dataset: 184 
Site-years in modelling dataset: 1180 


In [2]:
# ------------------------------------------------------------
# 2) Build zip index for FLUXMET HH files
# ------------------------------------------------------------

all_zip_files <- list.files(
  base_dir,
  pattern = "\\.zip$",
  recursive = TRUE,
  full.names = TRUE
)

zip_index_list <- list()
bad_zips <- character()

for (z in all_zip_files) {
  
  files_inside <- tryCatch(
    unzip(z, list = TRUE)$Name,
    error = function(e) {
      bad_zips <<- c(bad_zips, z)
      NULL
    }
  )
  
  if (is.null(files_inside)) next
  
  hh_file <- files_inside[
    str_detect(files_inside, "FLUXNET_FLUXMET_HH") &
      str_detect(files_inside, "\\.csv$")
  ]
  
  if (length(hh_file) > 0) {
    zip_index_list[[length(zip_index_list) + 1]] <-
      tibble(zip_path = z, hh_file = hh_file)
  }
}

zip_index <- bind_rows(zip_index_list) %>%
  mutate(SITE_ID = str_extract(hh_file, "[A-Z]{2}-[A-Za-z0-9]{3}")) %>%
  filter(!is.na(SITE_ID)) %>%
  distinct(SITE_ID, zip_path, hh_file, .keep_all = TRUE) %>%
  arrange(SITE_ID)

zip_index <- as.data.table(zip_index)

# keep only sites needed for current analysis
zip_index <- zip_index[SITE_ID %in% needed_site_years$SITE_ID]

fwrite(zip_index, file.path(out_dir, "fluxmet_hh_zip_index_used.csv"))
fwrite(data.table(zip_path = bad_zips), file.path(out_dir, "bad_or_corrupted_zip_files.csv"))

cat("Matched FLUXMET HH sites:", uniqueN(zip_index$SITE_ID), "\n")


Matched FLUXMET HH sites: 184 


In [3]:
# ------------------------------------------------------------
# 3) Function: calculate center of growing season for one site
# ------------------------------------------------------------

calc_cgs_one_site <- function(zip_path, hh_file, site_id) {
  
  cat("Processing:", site_id, "\n")
  
  # read csv inside zip
  hh <- tryCatch(
    fread(cmd = paste("unzip -p", shQuote(zip_path), shQuote(hh_file))),
    error = function(e) {
      message("Could not read: ", site_id)
      return(NULL)
    }
  )
  
  if (is.null(hh)) return(NULL)
  
  # choose GPP variable
  gpp_candidates <- c(
    "GPP_NT_VUT_REF",
    "GPP_DT_VUT_REF",
    "GPP_NT_CUT_REF",
    "GPP_DT_CUT_REF"
  )
  
  gpp_col <- gpp_candidates[gpp_candidates %in% names(hh)][1]
  
  if (is.na(gpp_col)) {
    message("No GPP column found for: ", site_id)
    return(NULL)
  }
  
  if (!("TIMESTAMP_START" %in% names(hh))) {
    message("No TIMESTAMP_START found for: ", site_id)
    return(NULL)
  }
  
  # timestamp
  hh[, timestamp := ymd_hm(as.character(TIMESTAMP_START))]
  hh[, date := as.Date(timestamp)]
  hh[, year := year(date)]
  hh[, doy := yday(date)]
  
  # GPP
  hh[, GPP := as.numeric(get(gpp_col))]
  hh[GPP < -1000, GPP := NA_real_]
  
  # daily GPP
  # using mean half-hourly GPP to avoid bias from incomplete days
  daily <- hh[, .(
    daily_GPP = mean(GPP, na.rm = TRUE),
    n_hh = sum(!is.na(GPP))
  ), by = .(year, date, doy)]
  
  # require at least half the day available
  daily <- daily[n_hh >= 24]
  
  if (nrow(daily) == 0) return(NULL)
  
  # calculate growing season metrics per year
  out <- daily[, {
    
    gpp_max <- max(daily_GPP, na.rm = TRUE)
    gpp_min <- min(daily_GPP, na.rm = TRUE)
    amp <- gpp_max - gpp_min
    threshold <- gpp_min + 0.3 * amp
    
    gs_days <- .SD[daily_GPP > threshold]
    
    if (nrow(gs_days) == 0 || is.na(amp) || amp <= 0) {
      
      list(
        GPPmin = gpp_min,
        GPPmax = gpp_max,
        GPP_amplitude = amp,
        GPP_threshold_30pct = threshold,
        GS_start_doy = NA_real_,
        GS_end_doy = NA_real_,
        GS_length_days = 0,
        CGS_midpoint_doy = NA_real_,
        CGS_weighted_doy = NA_real_
      )
      
    } else {
      
      # midpoint = center between first and last growing-season day
      gs_start <- min(gs_days$doy, na.rm = TRUE)
      gs_end <- max(gs_days$doy, na.rm = TRUE)
      gs_length <- length(unique(gs_days$doy))
      
      # weighted center = productivity-weighted center of season
      # weights are daily GPP above threshold
      weights <- gs_days$daily_GPP - threshold
      weights[weights < 0] <- 0
      
      weighted_center <- if (sum(weights, na.rm = TRUE) > 0) {
        sum(gs_days$doy * weights, na.rm = TRUE) / sum(weights, na.rm = TRUE)
      } else {
        mean(gs_days$doy, na.rm = TRUE)
      }
      
      list(
        GPPmin = gpp_min,
        GPPmax = gpp_max,
        GPP_amplitude = amp,
        GPP_threshold_30pct = threshold,
        GS_start_doy = gs_start,
        GS_end_doy = gs_end,
        GS_length_days = gs_length,
        CGS_midpoint_doy = mean(c(gs_start, gs_end)),
        CGS_weighted_doy = weighted_center
      )
    }
    
  }, by = year]
  
  out[, SITE_ID := site_id]
  out[, GPP_variable_used := gpp_col]
  
  setcolorder(out, c(
    "SITE_ID", "year", "GPP_variable_used",
    "GPPmin", "GPPmax", "GPP_amplitude", "GPP_threshold_30pct",
    "GS_start_doy", "GS_end_doy", "GS_length_days",
    "CGS_midpoint_doy", "CGS_weighted_doy"
  ))
  
  return(out)
}


In [ ]:
# ------------------------------------------------------------
# 4) Loop over sites with checkpoint
# ------------------------------------------------------------

cgs_list <- list()

for (i in seq_len(nrow(zip_index))) {
  
  site_id <- zip_index$SITE_ID[i]
  
  res <- calc_cgs_one_site(
    zip_path = zip_index$zip_path[i],
    hh_file = zip_index$hh_file[i],
    site_id = site_id
  )
  
  if (!is.null(res)) {
    cgs_list[[site_id]] <- res
    
    checkpoint <- rbindlist(cgs_list, fill = TRUE)
    
    fwrite(
      checkpoint,
      file.path(out_dir, "checkpoint_center_growing_season.csv")
    )
  }
}

In [ ]:
# keep only site-years in modelling dataset
cgs_dt <- merge(
  needed_site_years,
  cgs_dt,
  by.x = c("SITE_ID", "YEAR"),
  by.y = c("SITE_ID", "year"),
  all.x = TRUE
)
setnames(cgs_dt, "YEAR", "year")  # harmonize to lowercase for downstream code

fwrite(
  cgs_dt,
  file.path(out_dir, "center_growing_season_by_site_year.csv")
)

cat("Saved center of growing season table:\n")
cat(file.path(out_dir, "center_growing_season_by_site_year.csv"), "\n")

In [ ]:
# ------------------------------------------------------------
# 5) Merge back to modelling dataset
# ------------------------------------------------------------

model_dt_cgs <- merge(
  model_dt,
  cgs_dt[, .(
    SITE_ID, year,
    GS_start_doy,
    GS_end_doy,
    GS_length_days,
    CGS_midpoint_doy,
    CGS_weighted_doy
  )],
  by.x = c("SITE_ID", "YEAR"),
  by.y = c("SITE_ID", "year"),
  all.x = TRUE
)

fwrite(
  model_dt_cgs,
  file.path(out_dir, "EFP_mortality_trait_hydro_combined_with_CGS.csv")
)

cat("Saved modelling dataset with CGS:\n")
cat(file.path(out_dir, "EFP_mortality_trait_hydro_combined_with_CGS.csv"), "\n")